# 01 Build the slice caches, all cohorts (CPU, no GPU needed; A runs this)
Builder v2: every patient resampled to 2.0 mm slices x 0.7 mm pixels, centred on the tumour, clip
ceiling raised so the late phase doesn't saturate in the tumour (see `scripts/build_cache.py`).
1. Session options: Accelerator **None**, Internet **ON**, secret `GITHUB_TOKEN` attached.
2. **+ Add Input** -> all three raw Datasets (`mnj-raw-ispy1`, `mnj-raw-ispy2`, `mnj-raw-duke`;
   B and C must have shared theirs with you).
3. Quick check first: `LIMIT = 10`, run interactively, look at the grids. Then `LIMIT = 0` and
   **Save Version -> Save & Run All (Commit)** (several hours; keeps running with the browser closed).
4. When it finishes: check the QC tables and grids, then **New Dataset** from the output, named
   `mnj-cache-v2` (Private, shared with the team). It holds one folder per cohort, `cache_v2_<cohort>`.
   Training picks the highest builder version automatically if old caches are attached too.

**Builder v3** (`Z_MM = 0`): same as v2 in-plane (0.7 mm, tumour-centred), but the **native slices are
kept** (no resampling between slices). Output folders `cache_v3_<cohort>`; name the Dataset `mnj-cache-v3`.
Notebooks that compare v2 and v3 (`14_v3_enhancement`) choose the cache per run.

In [ ]:
# ==== PARAMETERS (edit these) ====
REPO = "AkulaReddy/MRI_chemo_response"
GIT_TAG = "d1.10"        # code version; A posts the tag to use
PERSON = ""             # set to "A", "B" or "C"
COHORTS = ["ispy1", "ispy2", "duke"]
LIMIT = 0               # 0 = all patients; 10 for a quick check
WORKERS = 4
RAW = {}                # {cohort: path} to override the automatic search for mnj-raw-<cohort>
Z_MM = 2.0              # 2.0 = builder v2 (2 mm slices); 0 = builder v3 (native slices, 0.7 mm in-plane)

In [ ]:
# ==== SETUP (do not edit) ====
import base64, json, os, shutil, subprocess, sys
assert PERSON in ("A", "B", "C"), 'set PERSON = "A", "B" or "C" in the parameters cell'
from kaggle_secrets import UserSecretsClient
_tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
_hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()

def git(*args, cwd="/tmp/repo"):
    """git with the token as a one-off header (never stored in .git/config or the output)."""
    r = subprocess.run(["git", "-c", f"http.extraHeader={_hdr}", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_tok, "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

shutil.rmtree("/tmp/repo", ignore_errors=True)
git("clone", "--quiet", "--depth", "1", "--branch", GIT_TAG, f"https://github.com/{REPO}.git", "/tmp/repo", cwd="/tmp")
os.chdir("/tmp/repo")
print("code at", GIT_TAG, git("log", "-1", "--format=%h %s"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
import time

def find_raw(cohort):
    """Kaggle may unpack the .tar.gz into a folder; either form works."""
    want = f"BreastDCEDL_{cohort.upper()}_min_crop"
    hits = []
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        if root.count("/") > 7:
            dirs[:] = []
            continue
        hits += [os.path.join(root, n) for n in dirs + files if n.startswith(want)]
        dirs[:] = [d for d in dirs if not d.startswith("BreastDCEDL_")]
    if not hits:
        return None
    return hits[0] if os.path.isdir(hits[0]) else os.path.dirname(hits[0])

raws = {c: RAW.get(c) or find_raw(c) for c in COHORTS}
print("raw inputs:", raws)
if not all(raws.values()):
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        print(root, dirs[:5], files[:5])
        if root.count("/") > 5:
            dirs[:] = []
    raise SystemExit(f"missing raw input for {[c for c, r in raws.items() if not r]}: attach mnj-raw-<cohort>")

OUTS, status = {}, {}
for c in COHORTS:   # one cohort at a time; a failed cohort is reported, the others still build
    OUTS[c] = f"/kaggle/working/cache_v{2 if Z_MM else 3}_{c}"
    cmd = [sys.executable, "scripts/build_cache.py", "--cohort", c, "--raw", raws[c], "--out", OUTS[c],
           "--workers", str(WORKERS), "--z-mm", str(Z_MM)] + (["--limit", str(LIMIT)] if LIMIT else [])
    t = time.time()
    print(f"\n===== {c}: {' '.join(cmd)}", flush=True)
    status[c] = subprocess.run(cmd).returncode
    print(f"===== {c}: exit {status[c]}, {(time.time() - t) / 60:.1f} min", flush=True)
print("\nexit codes:", status)

In [ ]:
import pandas as pd
from IPython.display import Image, display

for c, out in OUTS.items():
    print(f"\n==================== {c}")
    if not os.path.exists(f"{out}/cache_index.csv"):
        print("no cache_index.csv: the build failed, see the log above")
        continue
    idx = pd.read_csv(f"{out}/cache_index.csv")
    meta = json.load(open(f"{out}/cache_meta.json"))
    print({k: meta.get(k) for k in ("builder_version", "grid_mm", "git", "status_counts", "seconds")})
    for col in ("z_source", "xy_source"):
        if col in idx:
            print(col, idx[col].value_counts(dropna=False).to_dict())
    if "mask_csv_iou_yx" in idx and idx.mask_csv_iou_yx.notna().any():
        print("mask vs CSV box IoU:", idx.mask_csv_iou_yx.describe().round(3).to_dict())
    ok = idx[idx.status == "ok"]
    show = [k for k in ("src_dxy", "src_dz", "n_slices", "sat_frac") if k in ok]
    print("per patient (ok rows):")
    display(ok[show].describe(percentiles=[0.5]).round(3))
    if "sat_frac" in ok:
        print(f"tumour-box pixels with a saturated channel: median {ok.sat_frac.median():.4f}, "
              f"patients over 1%: {(ok.sat_frac > 0.01).sum()} of {len(ok)}")
    mb = sum(os.path.getsize(os.path.join(out, f)) for f in os.listdir(out)) / 1e6
    print(f"size on disk: {mb:.0f} MB")
    err = idx[idx.status == "error"]
    if len(err):
        display(err[["pid", "note"]].head(20))
    if os.path.exists(f"{out}/summary_grid.png"):
        display(Image(f"{out}/summary_grid.png"))